<a href="https://colab.research.google.com/github/kathir1406/AI-Powered-Weed-Detection/blob/main/AI_Agriculture_Doctor_Gemini_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌱 AI Agriculture Doctor
### Gemini Vision + Gradio | Google Colab

Upload a leaf photo or describe plant symptoms. Gemini will provide:
- Likely disease or issue, with confidence/uncertainty explained
- Simple explanation of visible symptoms
- **3 practical first steps** for treatment
- Prevention advice and when to contact an agricultural expert

> **Important:** This is an AI decision-support tool, not a laboratory diagnosis. A photo may not distinguish disease from nutrient deficiency, pests, herbicide injury, or environmental stress. Do not apply pesticides solely from an AI result; follow local label directions and agricultural extension advice.


In [ ]:
!pip -q install --upgrade google-genai gradio pillow

## 1. Enter your Gemini API key

Create a key at [Google AI Studio](https://aistudio.google.com/apikey). The key is requested privately at runtime and is not saved in this notebook.

In [1]:
import getpass
from google import genai

GEMINI_API_KEY = getpass.getpass("Enter your Gemini API key: ").strip()
if not GEMINI_API_KEY:
    raise ValueError("A Gemini API key is required.")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_NAME = "gemini-flash-latest"

print("Gemini client initialized. Model:", MODEL_NAME)

Enter your Gemini API key: ··········
Gemini client initialized. Model: gemini-flash-latest


## 2. AI diagnosis engine

The prompt asks Gemini to distinguish likely infectious disease from non-infectious causes, avoid pretending certainty, and give exactly three actionable first steps.

In [2]:
from PIL import Image
import io

SYSTEM_GUIDANCE = """
You are AI Agriculture Doctor, a cautious plant-health assistant for farmers.
Analyze the uploaded plant/leaf photo and the farmer's description.

Rules:
1. First say whether the image is clear enough and whether a plant/leaf is visible.
2. Give the most likely issue, but clearly label it as a preliminary possibility, not a confirmed diagnosis.
3. Consider fungal, bacterial, viral, pest, nutrient-deficiency, watering, heat, and chemical injury causes.
4. If evidence is weak or multiple causes look similar, say so and list up to 2 alternatives.
5. Do not invent a disease, crop, pathogen, or certainty percentage.
6. Give exactly 3 quick, practical first steps. Prefer low-risk steps such as isolating affected plants,
   removing severely affected leaves with clean tools, checking drainage/irrigation, and inspecting
   nearby plants. Do not prescribe pesticide brands, exact chemical doses, or unsafe mixtures.
7. If a pesticide/fungicide may be needed, advise using only a locally registered product for that crop
   and disease, following its label and local agricultural extension guidance.
8. Ask for a clearer photo or more details when needed (whole plant, leaf top and underside, stem,
   crop name, location, recent weather, irrigation, and spread pattern).
9. Keep language simple and farmer-friendly. Answer in the requested language.
10. Mention urgent red flags: rapid spread, severe wilting, stem/root rot, or suspected regulated pest.
"""

def diagnose_plant(image, crop_name, symptoms, location, language):
    if image is None and not (symptoms or "").strip():
        return "## Please upload a leaf photo or describe the symptoms."

    details = f"""
Crop/plant: {crop_name or 'Not provided'}
Farmer's symptom description: {symptoms or 'Not provided'}
Location: {location or 'Not provided'}
Response language: {language or 'English'}

Return the answer with these headings:
## 🌿 Preliminary diagnosis
## 🔎 Why this may be happening
## 🛠️ 3 quick steps to take now
1. ...
2. ...
3. ...
## 🛡️ Prevention
## 📸 What to check next
## ⚠️ Important safety note

If the photo does not support a reliable diagnosis, say "Insufficient visual evidence" and explain
which additional views/details are needed. Do not force a disease label.
"""

    contents = [SYSTEM_GUIDANCE + "\\n\\n" + details]
    if image is not None:
        # Convert to a supported RGB image and keep the upload reasonably small.
        img = image.convert("RGB")
        img.thumbnail((1600, 1600))
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=88)
        contents.append(types.Part.from_bytes(data=buf.getvalue(), mime_type="image/jpeg"))

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=contents
        )
        return response.text or "Gemini returned an empty response. Please try again."
    except Exception as exc:
        return f"### Diagnosis request failed\n\n{type(exc).__name__}: {exc}\n\nCheck your API key, internet connection, model availability, and API quota."

from google.genai import types
print("Diagnosis engine is ready.")

Diagnosis engine is ready.


## 3. Launch the interactive Agriculture Doctor

The interface supports image upload, crop name, symptom description, location, and English/Tamil output.

In [ ]:
import gradio as gr

custom_css = """
.gradio-container {max-width: 1100px !important; margin: auto !important;}
#hero {text-align:center; padding: 18px; border-radius: 18px;}
#diagnosis {border-radius: 14px;}
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="green", secondary_hue="lime"), css=custom_css) as demo:
    gr.Markdown(
        """
        <div id="hero">
        <h1>🌱 AI Agriculture Doctor</h1>
        <p>Smart plant health assistance powered by Google Gemini</p>
        <p><b>Upload a leaf photo • Describe symptoms • Get 3 practical steps</b></p>
        </div>
        """
    )
    with gr.Row():
        with gr.Column(scale=1):
            leaf_image = gr.Image(type="pil", label="📷 Upload plant / leaf photo")
            crop = gr.Textbox(label="🌾 Crop name", placeholder="e.g., Tomato, Paddy, Chilli, Cotton")
            symptoms = gr.Textbox(
                label="📝 Describe symptoms",
                placeholder="e.g., Yellow spots on older leaves, leaves curling, plant wilting...",
                lines=4
            )
            location = gr.Textbox(label="📍 Location (optional)", placeholder="e.g., Tirunelveli, Tamil Nadu")
            language = gr.Dropdown(
                choices=["English", "Tamil"],
                value="English",
                label="🌐 Response language"
            )
            submit = gr.Button("🔍 Diagnose Plant", variant="primary")
            clear = gr.Button("Clear")
        with gr.Column(scale=1):
            result = gr.Markdown(
                "### Your plant health report will appear here.\n\nUpload a photo or enter symptoms, then click **Diagnose Plant**.",
                elem_id="diagnosis"
            )
    submit.click(
        fn=diagnose_plant,
        inputs=[leaf_image, crop, symptoms, location, language],
        outputs=result
    )
    clear.click(
        fn=lambda: (None, "", "", "", "English", "Your next diagnosis will appear here."),
        inputs=[],
        outputs=[leaf_image, crop, symptoms, location, language, result]
    )
    gr.Markdown(
        """
        ---
        **Disclaimer:** AI-generated guidance can be wrong. Confirm serious or spreading problems with
        a local agricultural officer, KVK, or plant pathologist. Follow pesticide labels and local rules.
        """
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_1085/2511969470.py:9: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="green", secondary_hue="lime"), css=custom_css) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bcbf299d3dd6848706.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 4. Test prompts

Try one of these descriptions:
- **Tomato:** "Brown circular spots with yellow rings on older leaves."
- **Paddy:** "Leaf tips are turning yellow and drying."
- **Chilli:** "Leaves are curling and becoming smaller."
- **General:** "The lower leaves are yellow, but the new leaves are green."

A text description alone is not enough to confirm a disease. Add clear photos of both sides of affected leaves and the whole plant for better analysis.